# 02 — Preprocess: audio extraction, silent video, ASR transcripts

For each video in the manifest:
1. Extract 16-kHz mono audio (`.wav`) — used by S2/S4-S8.
2. Strip audio to make a silent video — used by S3.
3. Whisper-small transcribe → text — used by S5 (matched) and S7 (donor pool).
4. Build the qa_id → donor video mismatched-transcript pairing — used by S7.

**Runtime:** ~5-15 min for 600 samples (Whisper on GPU). Cached: rerun to skip.
**GPU:** strongly recommended for Whisper.


In [ ]:
# ─── Colab bootstrap ────────────────────────────────────────
# Mount Drive (caches model weights + videos across sessions),
# clone the repo, install dependencies, and configure the cache dirs.
import os, sys, subprocess, pathlib

IS_COLAB = "google.colab" in sys.modules

if IS_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_ROOT = "/content/drive/MyDrive/avut"
    REPO_DIR = "/content/omnimodel-research"
    os.makedirs(DRIVE_ROOT, exist_ok=True)
    if not os.path.exists(REPO_DIR):
        subprocess.run([
            "git", "clone",
            "https://github.com/samadasyed/omnimodel-research.git",
            REPO_DIR,
        ], check=True)
else:
    DRIVE_ROOT = os.path.expanduser("~/avut")
    REPO_DIR = str(pathlib.Path.cwd())
    os.makedirs(DRIVE_ROOT, exist_ok=True)

os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

# Persistent storage layout (data + model caches under DRIVE_ROOT)
DATA_DIR        = os.path.join(DRIVE_ROOT, "data")
VIDEO_DIR       = os.path.join(DATA_DIR, "videos")
AUDIO_DIR       = os.path.join(DATA_DIR, "audio")
SILENT_DIR      = os.path.join(DATA_DIR, "silent")
TRANSCRIPT_DIR  = os.path.join(DATA_DIR, "transcripts")
ANNOTATION_DIR  = os.path.join(DATA_DIR, "annotations")
RESULTS_DIR     = os.path.join(DRIVE_ROOT, "results")
RAW_PRED_DIR    = os.path.join(RESULTS_DIR, "raw_predictions")
METRICS_DIR     = os.path.join(RESULTS_DIR, "metrics")
HF_CACHE        = os.path.join(DRIVE_ROOT, ".cache", "hf")
WHISPER_CACHE   = os.path.join(DRIVE_ROOT, ".cache", "whisper")

for d in [VIDEO_DIR, AUDIO_DIR, SILENT_DIR, TRANSCRIPT_DIR,
          ANNOTATION_DIR, RAW_PRED_DIR, METRICS_DIR, HF_CACHE, WHISPER_CACHE]:
    os.makedirs(d, exist_ok=True)

# Redirect HF cache to Drive so we don't redownload weights each session
os.environ["HF_HOME"] = HF_CACHE
os.environ["TRANSFORMERS_CACHE"] = HF_CACHE
os.environ["HF_DATASETS_CACHE"] = HF_CACHE

print(f"Repo:       {REPO_DIR}")
print(f"Drive root: {DRIVE_ROOT}")
print(f"HF cache:   {HF_CACHE}")


In [ ]:
# ─── Install model dependencies ──────────────────────────
# transformers (Gemma-3n preview support landed in 4.51), accelerate,
# soundfile for audio I/O, openai-whisper for ASR. cv2 (opencv-python)
# is preinstalled on Colab and used for frame sampling.
%pip install -q -U "transformers>=4.51.0" "accelerate>=0.30"     "huggingface-hub>=0.24" "soundfile>=0.12" scipy
%pip install -q openai-whisper
# Qwen-Omni preview branch is only needed for notebook 05; install there.

# Gemma-3n is GATED on HuggingFace — you must accept the license at
# https://huggingface.co/google/gemma-3n-E2B-it and authenticate.
# In Colab: Tools → Secrets → add HF_TOKEN, then:
import os
if "HF_TOKEN" in os.environ or "HUGGINGFACE_TOKEN" in os.environ:
    from huggingface_hub import login
    login(token=os.environ.get("HF_TOKEN") or os.environ.get("HUGGINGFACE_TOKEN"))
    print("HF authenticated")
else:
    try:
        from google.colab import userdata
        tok = userdata.get("HF_TOKEN")
        if tok:
            from huggingface_hub import login
            login(token=tok)
            print("HF authenticated (from Colab Secrets)")
        else:
            print("WARN: no HF_TOKEN — Gemma-3n download will fail. "
                  "Add HF_TOKEN to Colab Secrets first.")
    except Exception:
        print("WARN: not in Colab and no HF_TOKEN env var. "
              "Run `huggingface-cli login` before loading Gemma.")


In [ ]:
import json, subprocess
from pathlib import Path

# ffmpeg ships with imageio[ffmpeg] but Colab has it system-wide too.
subprocess.run(["ffmpeg", "-version"], capture_output=True, check=True)
print("ffmpeg OK")

manifest_path = os.path.join(DATA_DIR, "sample_manifest_available.json")
with open(manifest_path) as f:
    samples = json.load(f)
print(f"Manifest: {len(samples)} samples")


## Audio extraction + silent video

In [ ]:
def extract_audio(video, out_wav):
    if os.path.exists(out_wav) and os.path.getsize(out_wav) > 0:
        return True
    r = subprocess.run([
        "ffmpeg", "-y", "-i", video,
        "-vn", "-acodec", "pcm_s16le", "-ar", "16000", "-ac", "1",
        out_wav,
    ], capture_output=True)
    return r.returncode == 0 and os.path.exists(out_wav)


def strip_audio(video, out_silent):
    if os.path.exists(out_silent) and os.path.getsize(out_silent) > 0:
        return True
    r = subprocess.run([
        "ffmpeg", "-y", "-i", video, "-an", "-c:v", "copy", out_silent,
    ], capture_output=True)
    return r.returncode == 0 and os.path.exists(out_silent)


stats = {"audio_ok": 0, "silent_ok": 0, "fail": 0}
for i, s in enumerate(samples, 1):
    vid = s["video_id"]
    video = os.path.join(VIDEO_DIR, f"{vid}.mp4")
    audio_out = os.path.join(AUDIO_DIR, f"{vid}.wav")
    silent_out = os.path.join(SILENT_DIR, f"{vid}_silent.mp4")
    if not os.path.exists(video):
        stats["fail"] += 1
        continue
    if extract_audio(video, audio_out):
        stats["audio_ok"] += 1
    else:
        stats["fail"] += 1
    if strip_audio(video, silent_out):
        stats["silent_ok"] += 1
    if i % 50 == 0:
        print(f"  [{i}/{len(samples)}] audio={stats['audio_ok']} silent={stats['silent_ok']}")
print(stats)


## Whisper transcripts (small)

In [ ]:
import whisper as whisper_lib
from src.transcript_utils import transcribe_with_whisper_python

print("Loading Whisper-small...")
whisper_model = whisper_lib.load_model("small", download_root=WHISPER_CACHE)
print("OK")

t_stats = {"ok": 0, "fail": 0}
for i, s in enumerate(samples, 1):
    vid = s["video_id"]
    audio_path = os.path.join(AUDIO_DIR, f"{vid}.wav")
    out_txt = os.path.join(TRANSCRIPT_DIR, f"{vid}.txt")
    if not os.path.exists(audio_path):
        t_stats["fail"] += 1
        continue
    try:
        transcribe_with_whisper_python(audio_path, out_txt, whisper_model)
        t_stats["ok"] += 1
    except Exception as e:
        print(f"  ✗ {vid}: {e}")
        t_stats["fail"] += 1
    if i % 50 == 0:
        print(f"  [{i}/{len(samples)}] {t_stats}")
print(t_stats)

# Free Whisper before model loads in subsequent notebooks
import gc, torch
del whisper_model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()


## Build mismatched-transcript pairing

For S7: each qa_id is paired with a *different* video in the same task type. The donor's transcript is what we'll inject.

In [ ]:
from src.transcript_utils import build_mismatched_pairs

pairs = build_mismatched_pairs(samples, seed=42)
mm_path = os.path.join(DATA_DIR, "mismatched_pairs.json")
with open(mm_path, "w") as f:
    json.dump({str(k): v for k, v in pairs.items()}, f, indent=2)
print(f"Mismatched-pair manifest: {mm_path}")
print(f"Pairings: {len(pairs)}")
print(f"Sample pairings (first 5):")
for qa_id, donor in list(pairs.items())[:5]:
    print(f"  qa_id={qa_id} → donor video {donor}")


Next: `03_pilot.ipynb` for a 24-sample sanity check, or skip to `04_main_eval_gemma.ipynb` for the full run.